# Solutions — Bakehouse PII and Security (Hard 06)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")
franchises   = spark.table("samples.bakehouse.sales_franchises")
reviews      = spark.table("samples.bakehouse.media_customer_reviews")


## Solution 1 — Advanced PII Masking

In [ ]:
result_1 = (
    transactions
    .join(customers,  on="customerID")
    .join(franchises.select("franchiseID", F.col("name").alias("franchiseName"),
                             "city", "country"), on="franchiseID")
    .withColumn("firstName",  F.lit("***"))
    .withColumn("lastName",   F.lit("***"))
    .withColumn("emailAddress",
        F.concat(
            F.substring("emailAddress", 1, 1),
            F.lit("***@"),
            F.regexp_extract("emailAddress", r"@(.+)$", 1)
        )
    )
    .withColumn("phoneNumber",
        F.concat(F.lit("***-***-"),
                 F.substring(F.regexp_replace("phoneNumber", r"[^0-9]", ""), -3, 3))
    )
    .withColumn("maskedCard",
        F.concat(F.lit("****-****-****-"),
                 F.substring(F.cast("cardNumber", "string"), -4, 4))
    )
    .select(
        "transactionID", "dateTime", "franchiseName", "firstName", "lastName",
        "emailAddress", "phoneNumber", "city", "country", "product",
        "quantity", "totalPrice", "paymentMethod", "maskedCard"
    )
)


## Solution 2 — Email Domain Analysis

In [ ]:
result_2 = (
    customers
    .withColumn("email_domain", F.regexp_extract("emailAddress", r"@(.+)$", 1))
    .groupBy("email_domain")
    .agg(F.count("*").alias("customer_count"))
    .orderBy(F.col("customer_count").desc())
    .limit(10)
)


## Solution 3 — Customer Deduplication

In [ ]:
result_3 = (
    customers
    .groupBy("firstName", "lastName", "country")
    .agg(
        F.count("*").alias("duplicate_count"),
        F.collect_list("customerID").alias("customer_ids"),
    )
    .filter(F.col("duplicate_count") > 1)
    .withColumnRenamed("firstName", "first_name")
    .withColumnRenamed("lastName",  "last_name")
    .orderBy(F.col("duplicate_count").desc())
)


## Solution 4 — Payment Card Anomaly Detection

In [ ]:
result_4 = (
    transactions
    .groupBy("cardNumber")
    .agg(
        F.countDistinct("customerID").alias("distinct_customers"),
        F.count("*").alias("transaction_count"),
    )
    .filter(F.col("distinct_customers") > 3)
    .withColumn("masked_card",
        F.concat(F.lit("****"), F.substring(F.cast("cardNumber", "string"), -4, 4))
    )
    .select("masked_card", "distinct_customers", "transaction_count")
    .orderBy(F.col("distinct_customers").desc())
)


## Solution 5 — Transaction Velocity Anomaly

In [ ]:
w_1h = Window.partitionBy("customerID").orderBy("dateTime").rangeBetween(-3600, 0)

result_5 = (
    transactions
    .withColumn("ts", F.unix_timestamp("dateTime"))
    .withColumn("transactions_in_last_hour",
        F.count("*").over(
            Window.partitionBy("customerID")
                  .orderBy(F.unix_timestamp("dateTime"))
                  .rangeBetween(-3600, 0)
        )
    )
    .filter(F.col("transactions_in_last_hour") >= 3)
    .select("customerID", "dateTime", "transactions_in_last_hour")
    .orderBy("customerID", "dateTime")
)


## Solution 6 — Revenue Reconciliation Audit

In [ ]:
result_6 = (
    transactions
    .withColumn("expected_price", F.col("unitPrice") * F.col("quantity"))
    .withColumn("discrepancy",    F.col("totalPrice") - F.col("expected_price"))
    .filter(F.col("discrepancy") != 0)
    .select(
        "transactionID", "product",
        F.col("unitPrice").alias("unitPrice"),
        "quantity",
        F.col("totalPrice").alias("totalPrice"),
        F.round("expected_price", 2).alias("expected_price"),
        F.round("discrepancy", 2).alias("discrepancy"),
    )
    .orderBy(F.abs("discrepancy").desc())
)


## Solution 7 — Geographic Revenue Heatmap

In [ ]:
result_7 = (
    transactions
    .join(franchises.select("franchiseID", "lat", "lon"), on="franchiseID")
    .withColumn("lat_bucket", F.round(F.col("lat"), 1))
    .withColumn("lon_bucket", F.round(F.col("lon"), 1))
    .groupBy("lat_bucket", "lon_bucket")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.countDistinct("customerID").alias("unique_customers"),
    )
    .orderBy(F.col("total_revenue").desc())
)


## Solution 8 — Data Quality Report

In [ ]:
table_map = {
    "sales_transactions":        (transactions, "transactionID"),
    "sales_customers":           (customers,    "customerID"),
    "sales_franchises":          (franchises,   "franchiseID"),
    "media_customer_reviews":    (reviews,      "new_id"),
}

rows = []
for tname, (df, _pk) in table_map.items():
    row_count    = df.count()
    col_count    = len(df.columns)
    null_cols    = [c for c in df.columns if df.filter(F.col(c).isNull()).count() > 0]
    null_str     = ",".join(null_cols) if null_cols else ""
    rows.append((tname, row_count, col_count, null_str))

result_8 = spark.createDataFrame(rows,
    ["table_name", "row_count", "column_count", "columns_with_nulls"])
